# Lab Sidekick — GPT explain, Llama example

Week 1 exercise: one technical question, two models in a **pipeline** (not a bake-off).

1. **GPT** — analogy + correct *what/why* (frontier is stronger at naming the idiom)
2. **Llama (Ollama)** — a small runnable snippet that follows GPT’s explanation

If Ollama is not running, open a terminal and run `ollama serve`. Smaller machines can use `llama3.2:1b` instead of `llama3.2`.


In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [ ]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
# set up environment — OpenAI cloud + Ollama (same OpenAI-compatible client as day 2)

load_dotenv(override=True)
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

print("Ollama probe:", requests.get("http://localhost:11434", timeout=5).content)


In [ ]:
# pull llama3.2 once if you don't have it yet (day 2 pattern)
# !ollama pull llama3.2


In [ ]:
# GPT explains; Llama writes a snippet from that explanation (pipeline)

GPT_SYSTEM = """
You explain one technical idea for an LLM Engineering student.
Write markdown with EXACTLY these headings:

### Analogy
3-4 lines. A concrete picture that maps onto the code (each part of the analogy = a part of the syntax). No extra plot.

### Concepts
Teach the ideas BEFORE restating the snippet. Short bullets:
- What a generator is vs returning a list (yield pauses; return ends).
- What `yield from iterable` does (delegate: like for x in iterable: yield x).
- Name every construct in the question exactly (set vs dict vs list comprehension; .get vs []).
If those constructs are present, say what uniqueness/filtering/`None` handling they do.
Do not claim a set comprehension is lazy — it builds the set first, then yield from walks it.

### How this snippet uses them
Quote the code. 3-5 sentences connecting each piece of syntax to the concepts above.
Remind that `yield from` must live inside a `def` generator.

### Pitfall
One bullet. The mistake students actually make.

Hard rules: under 250 words. No full programs. No invented APIs.
""".strip()

LLAMA_SYSTEM = """
You turn an explanation into one tiny runnable example.
Hard rules:
- One short sentence, then ONE fenced python block. Nothing after the block.
- Put the question's code inside a function if it uses yield/yield from.
- Copy identifiers and methods from the question (.get vs [], yield from vs return).
- 2-4 sample inputs. A for-loop that prints. Optional one-line comments only.
- Max 20 lines of Python. No unused imports, no classes unless required.
""".strip()


def stream_answer(client: OpenAI, model: str, messages: list[dict], heading: str) -> str:
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        temperature=0.2,
    )
    text = f"## {heading}\n\n"
    handle = display(Markdown(text), display_id=True)
    for chunk in stream:
        text += chunk.choices[0].delta.content or ""
        update_display(Markdown(text), display_id=handle.display_id)
    return text


def explanation_body(full_markdown: str) -> str:
    """Drop our ## heading so Llama sees only GPT's explanation."""
    lines = full_markdown.splitlines()
    if lines and lines[0].startswith("## "):
        return "\n".join(lines[1:]).strip()
    return full_markdown.strip()


def lab_sidekick(question: str):
    gpt_md = stream_answer(
        openai,
        MODEL_GPT,
        [
            {"role": "system", "content": GPT_SYSTEM},
            {"role": "user", "content": question.strip()},
        ],
        f"1. Concept — GPT `{MODEL_GPT}`",
    )
    llama_user = (
        "Write the example now.\n"
        "Wrap the question's code in a small function and demo it.\n\n"
        f"QUESTION:\n{question.strip()}\n\n"
        f"EXPLANATION (follow this; do not contradict):\n{explanation_body(gpt_md)}"
    )
    stream_answer(
        ollama,
        MODEL_LLAMA,
        [
            {"role": "system", "content": LLAMA_SYSTEM},
            {"role": "user", "content": llama_user},
        ],
        f"2. Example — Llama `{MODEL_LLAMA}`",
    )


In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# pipeline: GPT explains, then Llama gets that text and writes an example

lab_sidekick(question)


In [ ]:
# try another course-shaped question

lab_sidekick(
    "What does stream=True do on chat.completions.create, and how is delta.content different from message.content?"
)
